# 05: The math engine: optimisation, recurrences and reading models

**Practice set:** `practice/05-math-optimization.md` · **Exam tasks:** Jan 18, Recursive V-67 Outbreak (pp. 4–5), Phulera Godown (pp. 6–7), Crude Curve Forecast (pp. 7–8), Catalyst Constraint (pp. 8–9) · **Moves:** M6, M7, M9, M13

## Hook

**Scene 1.** Five of your robots work along one long warehouse aisle, and you can install **one** charging dock. Where should it go? "Wherever is best" isn't an answer until you say *best for what*:
- the least **total** driving,
- the least **worst-case** driving for the unluckiest robot, or
- something that punishes long trips extra hard.

Each choice gives a different spot, sometimes wildly different. By the end of section 4 you'll know all three answers without trying a single position.

**Scene 2.** Your Arduino Uno Q gets a fixed compute budget every control cycle, and you split it between **perception** (the camera model) and **planning**. Every extra millisecond helps a bit less than the one before. How should you split it? And here's the trap: is spending the *whole* budget even right?

This notebook is the math engine behind several exam tasks. You'll learn to find the best choice (optimisation), predict how a quantity grows day by day (recurrences), and read a model's numbers straight off a graph.

## Before you start

- **You need:** algebra, plotting a parabola, and the MAE vs RMSE plot from notebook 01, section 7. Everything else (slopes, logs, sines, bases) is built here from scratch.
- **How to use this notebook:** run the cells in order. When you see **PREDICT**, write your guess *before* running the next cell. When you see **BUILD**, write the function yourself; the test cell tells you when it's right. Hints sit in collapsed blocks. Open one only after a real attempt.
- **Pen and paper stay on the desk.** The exam has no computer. The notebook is for checking and exploring *after* you've reasoned it out.

In [ ]:
import math
import prep
import numpy as np
import matplotlib.pyplot as plt

---
## 1. argmin is a search

**Idea.** "Find the best x" means: try candidates, score each one, and keep the one with the smallest score. Two words to keep apart:
- **min** f is the smallest *value* f reaches.
- **argmin** f is the *input* x that reaches it ("the argument that gives the minimum").

Example: f(x) = (x − 3)² + 1. Its min is 1 and its argmin is 3.

On a computer you can always brute-force an argmin by trying many candidates. On paper you need a *reason* that tells you the answer directly. This section does the brute force; sections 2–4 find the reasons.

**Smallest example.** Three robots at positions 1, 2 and 6. Try a dock at every whole-number position from 0 to 7, and score each position three ways.

In [ ]:
robots = [1, 2, 6]
rows = []
for x in range(0, 8):
    total = sum(abs(x - a) for a in robots)
    squares = sum((x - a) ** 2 for a in robots)
    worst = max(abs(x - a) for a in robots)
    rows.append([x, total, squares, worst])
prep.table(rows, header=["dock x", "sum |x-a| (total)", "sum (x-a)^2", "max |x-a| (worst)"])

**PREDICT:** for each column, which x wins? Do all three columns agree? Look hard at the last column: is there a clear single winner?

### BUILD 1: `argmin_brute`
Return the candidate with the smallest `f(candidate)`. If several tie, return the **first** one in the list.

In [ ]:
def argmin_brute(f, candidates):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(argmin_brute, [
    ((lambda x: (x - 3) ** 2, [0, 1, 2, 3, 4]), 3),
    ((abs, [-2, 3, 1]), 1),
    ((lambda x: 0, [5, 6]), 5),
    ((lambda x: -x, [1, 2]), 2),
])

### EXPLORE: a finer search
The worst-case column tied at x = 3 and x = 4. **PREDICT:** if you search in steps of 0.01 instead of whole numbers, where does each loss bottom out?

In [ ]:
if prep.ready(argmin_brute, abs, [1]):
    grid = [round(v, 2) for v in np.linspace(-1, 8, 901)]
    losses = {
        "sum |x-a|": lambda x: sum(abs(x - a) for a in robots),
        "sum (x-a)^2 / 10": lambda x: sum((x - a) ** 2 for a in robots) / 10,
        "max |x-a|": lambda x: max(abs(x - a) for a in robots),
    }
    plt.figure(figsize=(6, 3.5))
    for name, f in losses.items():
        best = argmin_brute(f, grid)
        plt.plot(grid, [f(x) for x in grid], label=f"{name}: argmin {best}")
        plt.plot([best], [f(best)], "ko")
    for a in robots:
        plt.axvline(a, color="gray", lw=0.5)
    plt.xlabel("dock position x"); plt.ylim(0, 12); plt.legend(fontsize=8); plt.grid(alpha=.3)
    plt.title("Three losses, three different best docks"); plt.show()

Brute force only finds the best among the candidates **you tried**. The whole-number search missed the worst-case winner completely. That's one reason to want a *reason*, not just a search.

---
## 2. Why the median wins Σ|x − a|: nudge and count

**Idea.** Stand the dock at some x and nudge it right by a tiny step ε (a small positive number).
- Every robot to your **left** gets ε farther away: +ε each.
- Every robot to your **right** gets ε closer: −ε each.

So the total changes by ε × (number on the left − number on the right). Keep moving right while more robots are on your right, because each nudge then *lowers* the total. Stop when the two sides balance. That balance point is the **median**, the middle value when the positions are sorted.

**Smallest example.** Four robots this time, at 1, 2, 6, 7. For a dock at x, count robots on each side and look at the total.

In [ ]:
four = [1, 2, 6, 7]
rows = []
for x in [0, 1.5, 3, 4, 5, 6.5, 8]:
    left = sum(a < x for a in four)
    right = sum(a > x for a in four)
    rows.append([x, left, right, left - right, sum(abs(x - a) for a in four)])
prep.table(rows, header=["x", "robots left", "robots right", "change per nudge (left - right)", "total"])

**PREDICT:** with 4 robots there's no single middle one. Where does the total stop falling, and what does the bottom of the curve look like?

In [ ]:
xs = np.linspace(-1, 9, 501)
plt.figure(figsize=(6, 3))
plt.plot(xs, [sum(abs(x - a) for a in four) for x in xs])
for a in four:
    plt.axvline(a, color="gray", lw=0.5)
plt.xlabel("dock position x"); plt.ylabel("total distance"); plt.grid(alpha=.3)
plt.title("Four robots: a flat bottom"); plt.show()

Between the two middle robots (2 and 6), one robot's step farther is cancelled by another's step closer, so the total is flat. **Every** point in [2, 6] is an argmin. With an odd number of robots the flat part shrinks to a single point, the middle robot. (Remember the flat bottom. One exam task makes you choose a single point from it.)

**EXPLORE. PREDICT:** in the three-robot example [1, 2, 6], move the robot at 6 out to 600. Where does the total-distance dock go now?

In [ ]:
if prep.ready(argmin_brute, abs, [1]):
    far = [1, 2, 600]
    print("argmin of sum |x-a| for", far, "=", argmin_brute(lambda x: sum(abs(x - a) for a in far), range(0, 701)))

It doesn't move. The far robot is still "one robot on the right", however far away it is. The median only counts *how many* points are on each side, not how far. That's why the median is called **robust to outliers**.

---
## 3. Why the mean wins Σ(x − a)²: the vertex of a parabola

**Idea.** Expand the sum. Each term is (x − a)² = x² − 2ax + a². Adding n of them:

$$\sum (x - a_i)^2 = n\,x^2 - 2\Big(\sum a_i\Big)x + \sum a_i^2$$

That's a parabola px² + qx + r with p = n > 0, so it opens upwards and has one lowest point, the **vertex**.

**Where is the vertex? (No calculus needed.)** Complete the square:

$$p x^2 + q x + r = p\Big(x + \frac{q}{2p}\Big)^2 + \Big(r - \frac{q^2}{4p}\Big)$$

(Expand the right side to check it.) A square is never negative, so the right side is smallest exactly when the bracket is 0, at **x = −q/(2p)**.

For our sum, q = −2Σa and p = n, so the vertex is x = 2Σa/(2n) = Σa/n: the **mean**.

**Smallest example.** Robots 1, 2, 6.

In [ ]:
n, s1, s2 = len(robots), sum(robots), sum(a * a for a in robots)
p, q, r = n, -2 * s1, s2
print(f"sum (x-a)^2 = {p}x^2 + ({q})x + {r}")
print(f"vertex x = -q/(2p) = {-q}/{2 * p} = {-q / (2 * p)}")
print(f"mean of robots = {s1}/{n} = {s1 / n}")
print(f"completed square: {p}(x - {-q / (2 * p)})^2 + {r - q * q / (4 * p)}")

Check against the table in section 1: at x = 3 the squared column was 14, the smallest in the column. ✓

**Why the mean gets dragged around.** Squaring makes a robot 10 units away count 100, so far robots pull hard. Move the robot at 6 out to 600 and the mean jumps to about 201, while the median (section 2) stays at 2.

---
## 4. Why the midrange wins max|x − a|

**Idea.** The worst-case distance only ever depends on the two **extreme** robots, the leftmost and the rightmost. Every robot in between is closer to the dock than one of the extremes. Start at the leftmost robot, where the rightmost one is the problem. Moving right shrinks that distance and grows the distance to the leftmost robot. The worst case is smallest when the two are **equal**: at the midpoint of the extremes, (min + max)/2, the **midrange**.

**Smallest example.** Robots 1, 2, 6: the midrange is (1 + 6)/2 = 3.5, with worst-case distance 2.5. That's exactly what the fine search found and the whole-number search missed.

| Loss to minimise | Best dock | Depends on |
|---|---|---|
| Σ\|x − a\| (total) | median | only the *order* of the points |
| Σ(x − a)² | mean | every value, far ones most |
| max\|x − a\| (worst case) | midrange | only the two extremes |

This is move **M6: what does the argmin pick?**

**Now do practice problem 5.1 on paper**, then check:

In [ ]:
prep.check("5.1a", ...)   # x minimising the total distance
prep.check("5.1b", ...)   # x minimising the sum of squares
prep.check("5.1c", ...)   # x minimising the worst-case distance

---
## 5. Two dimensions: L1 distance splits into coordinates

**Idea.** On a grid of perpendicular aisles, a robot can't drive diagonally. The distance from (x, y) to (a, b) is the **Manhattan** or **L1 distance**: |x − a| + |y − b|. Add it up over all robots and **regroup** the terms:

$$\sum_i \big(|x - a_i| + |y - b_i|\big) = \underbrace{\sum_i |x - a_i|}_{\text{only depends on } x} + \underbrace{\sum_i |y - b_i|}_{\text{only depends on } y}$$

The two parts don't interact. Changing x can't change the second part, and changing y can't change the first. So you can minimise each part on its own: x = median of the a's, y = median of the b's. A problem that splits like this is called **separable**.

**Smallest example.** Robots at (0, 0), (4, 1) and (1, 5).

In [ ]:
pts = [(0, 0), (4, 1), (1, 5)]
print("x-coordinates sorted:", sorted(p[0] for p in pts), "-> median", sorted(p[0] for p in pts)[1])
print("y-coordinates sorted:", sorted(p[1] for p in pts), "-> median", sorted(p[1] for p in pts)[1])
grid2 = [(x, y) for x in range(-1, 7) for y in range(-1, 7)]
if prep.ready(argmin_brute, abs, [1]):
    print("brute force over a grid:", argmin_brute(lambda g: sum(abs(g[0] - a) + abs(g[1] - b) for a, b in pts), grid2))

Notice the best dock (1, 1) isn't any robot's position. Each coordinate is a median, but of *different* robots.

**PREDICT:** add a fourth robot at (6, 3). With an even number of robots, what shape is the set of best docks?

In [ ]:
pts4 = pts + [(6, 3)]
X, Y = np.meshgrid(np.linspace(-1, 7, 161), np.linspace(-1, 6, 141))
Z = sum(np.abs(X - a) + np.abs(Y - b) for a, b in pts4)
plt.figure(figsize=(5, 4))
cs = plt.contourf(X, Y, Z, levels=20, cmap="viridis")
plt.contour(X, Y, Z, levels=[Z.min() + 1e-6], colors="white")
plt.colorbar(cs, label="total L1 distance")
plt.scatter(*zip(*pts4), color="red", zorder=3, label="robots")
plt.legend(); plt.title("Four robots: where is the flat bottom?"); plt.show()

A **rectangle**: every x between the two middle x's, paired with every y between the two middle y's. That's the 1-D flat bottom from section 2, once per coordinate.

(Careful: ordinary straight-line distance √((x − a)² + (y − b)²) does **not** split like this, because the square root mixes x and y together. Separability is special to L1, and to the squared distance.)

---
## 6. Marginal gain, slope, and the budget check

**Idea.** Back to scene 2. Say perception's accuracy gain from h milliseconds is g₁(h) = 4h − h², and planning's is g₂(h) = 10h − h². Both show **diminishing returns**: each extra millisecond adds less than the last, and past a point it even hurts.

The **marginal gain** at h is the extra gain from one more tiny bit of budget, per unit of budget. Geometrically it's the **slope** of the curve at h: rise ÷ run over a tiny step ε.

**Smallest example.** Compute (g₁(h + ε) − g₁(h)) / ε for smaller and smaller ε.

In [ ]:
def g_perc(h):
    return 4 * h - h * h

def g_plan(h):
    return 10 * h - h * h

rows = []
for h in [0, 1, 2, 3]:
    rows.append([h] + [(g_perc(h + e) - g_perc(h)) / e for e in (0.1, 0.01, 0.001)])
prep.table(rows, header=["h", "eps = 0.1", "eps = 0.01", "eps = 0.001"])

**PREDICT:** what number is each row heading towards? Find a formula in h.

<details><summary>Check with algebra</summary>

g₁(h + ε) − g₁(h) = 4(h + ε) − (h + ε)² − 4h + h² = 4ε − 2hε − ε². Divide by ε: 4 − 2h − ε. As ε shrinks to nothing, the slope is **4 − 2h**. In general, the slope of ph² + qh + r is **2ph + q**. It's zero exactly at h = −q/(2p), which is the vertex from section 3 again: the bottom (or top) of a parabola is where it's flat.

</details>

This "slope from a tiny step" is what mathematicians call the **derivative**. You don't need any rules for it in this notebook, only the idea: **slope = how fast f changes per unit of input, and at the bottom of a valley (or top of a hill) the slope is 0.**

### Splitting a budget
You have B ms in total: h₁ + h₂ ≤ B. Suppose perception's marginal gain is bigger than planning's. Move a tiny ε from planning to perception: you lose about (slope₂)·ε and gain about (slope₁)·ε, a net win. So at the best split, **if the whole budget gets used**, the two marginal gains must be **equal**. (Unless one model gets nothing at all because even its first millisecond is worth less than the other's last.)

**PREDICT:** with B = 9 ms, should you spend all 9? Look at where each gain curve peaks before answering.

In [ ]:
hs = np.linspace(0, 9, 181)
plt.figure(figsize=(6, 3.5))
plt.plot(hs, g_perc(hs), label="perception: 4h - h^2")
plt.plot(hs, g_plan(hs), label="planning: 10h - h^2")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("ms given to this module"); plt.ylabel("accuracy gain"); plt.legend(); plt.grid(alpha=.3)
plt.show()

### BUILD 2: `best_split_budget`
Try every split h₁ = i·step and h₂ = j·step with h₁ + h₂ ≤ B (**not** forced to equal B). Return `(h1, h2, total)` for the largest g1(h1) + g2(h2). On a tie, keep the first one found, scanning h1 upwards and, for each h1, h2 upwards.

In [ ]:
def best_split_budget(g1, g2, B, step):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(best_split_budget, [
    ((g_perc, g_plan, 4, 0.5), (0.5, 3.5, 24.5)),
    ((g_perc, g_plan, 9, 0.5), (2.0, 5.0, 29.0)),
    ((lambda h: h, lambda h: 2 * h, 3, 1), (0, 3, 6)),
    ((lambda h: -h, lambda h: -h, 5, 1), (0, 0, 0)),
])

**Read the two results.**
- **B = 4 (the budget binds):** the peaks need 2 + 5 = 7 ms, but you only have 4. The best split uses all 4, and the slopes match: 4 − 2(0.5) = 3 and 10 − 2(3.5) = 3. ✓
- **B = 9 (the budget doesn't bind):** each module sits at its own peak (slope 0), and **2 ms stay unused**. Spending them would push a module past its peak and *lower* its gain.

**The struggle, on purpose.** A classmate insists "use the whole budget" for B = 9: they set h₁ + h₂ = 9, equate slopes (4 − 2h₁ = 10 − 2h₂), and get h₁ = 3, h₂ = 6, total 3 + 24 = 27. That's worse than 29. Equal marginal gains is the right rule *only when the budget binds*.

**Aha: check whether the budget binds first** (move **M7: equal marginal gains, check whether the constraint binds**). Find each use's own peak. If the peaks fit inside the budget, you're done. Only otherwise, spend it all and equalise the slopes.

**Now do practice problem 5.2 on paper**, then check:

In [ ]:
prep.check("5.2a", ...)   # max total gain with B = 10
prep.check("5.2b", ...)   # max total gain with B = 5
prep.check("5.2c", ...)   # the classmate's total gain

---
## 7. AM-GM: equal parts make the biggest product

**Idea.** Fix the **sum** of two non-negative numbers a + b. Their **product** ab is largest when they're equal.

The name: (a + b)/2 is the *arithmetic mean* (AM, the ordinary average) and √(ab) is the *geometric mean* (GM). The AM-GM inequality says AM ≥ GM.

**Smallest example.** a + b = 10.

In [ ]:
rows = [[a, 10 - a, (a + 10 - a) / 2, math.sqrt(a * (10 - a)), a * (10 - a)] for a in [0, 1, 2, 3, 4, 5, 6, 8]]
prep.table(rows, header=["a", "b", "AM (a+b)/2", "GM sqrt(ab)", "product ab"])

**Proof in one line.** A square is never negative, so for a, b ≥ 0:

$$(\sqrt a - \sqrt b)^2 \ge 0 \;\Longrightarrow\; a - 2\sqrt{ab} + b \ge 0 \;\Longrightarrow\; \frac{a+b}{2} \ge \sqrt{ab}$$

Equality holds only when √a = √b, i.e. a = b. Squaring both sides: ab ≤ ((a + b)/2)². So with the sum fixed, the product can never beat the square of half the sum, and it reaches it exactly when the parts are equal.

**Robot example.** A rectangular test pen with 40 m of fence on all four sides: 2x + 2y = 40, so x + y = 20, so xy ≤ 10² = 100. The best pen is a 10 × 10 square.

It works the other way too: if the **product** is fixed, the **sum** is smallest when the parts are equal. (A pad of area 36 has perimeter at least 2(6 + 6) = 24.)

**The struggle to watch for.** AM-GM only helps if the two things you apply it to have a **fixed sum**. In practice problem 5.3(a) the obvious pair doesn't. Ask: *which two quantities add up to something I know?*

---
## 8. Convexity and Jensen: why an ensemble helps

**Idea.** A function is **convex** if the chord between any two points on its graph lies **above** the graph (the curve "holds water", like x²).

**Smallest example.** f(x) = x², with the points at x = 1 and x = 5.

In [ ]:
xs = np.linspace(0, 6, 121)
plt.figure(figsize=(5, 3.5))
plt.plot(xs, xs ** 2, label="f(x) = x^2")
plt.plot([1, 5], [1, 25], "o--", label="chord from x=1 to x=5")
plt.plot([3], [13], "s", label="average of f: (1 + 25)/2 = 13")
plt.plot([3], [9], "^", label="f of average: 3^2 = 9")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("Convex: chord above the curve"); plt.show()

The middle of the chord is the **average of the f-values**. The curve below it is **f of the average**. For a convex f:

$$f(\text{average of the inputs}) \le \text{average of } f(\text{inputs})$$

This is **Jensen's inequality**. "Average first, then apply f" never beats "apply f, then average", when f is convex.

**Why this matters for ML.** Three models predict a value whose truth is 5. Their predictions are 3, 6 and 9. Squared error is convex, so compare:
- the **average member**'s squared error (apply f, then average), and
- the **ensemble**'s squared error, where the ensemble predicts the average of the three (average, then apply f).

In [ ]:
truth, preds = 5, [3, 6, 9]
errs = [pr - truth for pr in preds]
rows = [[pr, e, e * e] for pr, e in zip(preds, errs)]
prep.table(rows, header=["prediction", "error", "squared error"])
avg_member = sum(e * e for e in errs) / len(errs)
ens_pred = sum(preds) / len(preds)
print(f"average member's squared error = {avg_member}")
print(f"ensemble predicts {ens_pred}, squared error = {(ens_pred - truth) ** 2}")

**Why it's guaranteed, for squares, in two lines.** Let the errors have mean m. Then

$$\text{average of } e^2 = m^2 + \text{average of } (e - m)^2$$

(Expand (e − m)² = e² − 2em + m² and average: the middle term averages to −2m·m.) The last term is an average of squares, so it's ≥ 0. So the average member's error is the ensemble's error **plus the spread** of the errors. Here: 7 = 1 + 6. The more the models disagree, the more the ensemble gains.

**PREDICT:** is the ensemble always better than the **best** single member? Try to invent two models where it isn't.

In [ ]:
for errors in [[0, 10], [1, -2, 7], [2, 2, 2]]:
    m = sum(errors) / len(errors)
    print(f"errors {errors}: best member {min(e * e for e in errors)}, "
          f"average member {sum(e * e for e in errors) / len(errors):.2f}, ensemble {m * m:.2f}")

With errors [0, 10] the ensemble (25) is worse than the perfect member (0), but still better than the *average* member (50). Jensen promises "better than average", not "better than the best". And when all models make the same error, averaging gains nothing.

**Now do practice problem 5.3 on paper**, then check:

In [ ]:
prep.check("5.3a", ...)       # the maximum area in m^2
prep.check("5.3b_avg", ...)   # the average squared error of the three models
prep.check("5.3b_ens", ...)   # the squared error of their average prediction

---
## 9. Recurrences: tabulate, then shift to the fixed point

**Idea.** A **recurrence** gives tomorrow's value from today's, like x_{t+1} = 2x_t − 10. The first move is always the same: **tabulate**, one row per step, applying the rule exactly (move **M9: tabulate recurrences**).

**Smallest example.** A robot swarm: each day every robot builds one more robot, then 10 robots fail and are scrapped. So R_{t+1} = 2R_t − 10, starting from R₀ = 15.

In [ ]:
R_t = 15
rows = []
for t in range(6):
    rows.append([t, R_t, R_t - 10])
    R_t = 2 * R_t - 10
prep.table(rows, header=["day t", "R_t", "R_t - 10"])

**PREDICT:** look at the last column. What pattern do you see, and why is 10 the special number?

**The fixed point.** A **fixed point** is a value the rule leaves unchanged: x* = 2x* − 10, so x* = 10. If the swarm ever had exactly 10 robots, it would stay at 10 forever.

Now measure the distance from the fixed point, d_t = R_t − 10:

d_{t+1} = R_{t+1} − 10 = (2R_t − 10) − 10 = 2(R_t − 10) = 2d_t.

The +10 and −10 cancel, and the distance just **doubles** every day: d_t = 5·2^t, so R_t = 10 + 5·2^t. A sequence that multiplies by the same number each step is called **geometric**.

**In general**, for x_{t+1} = r·x_t + c: the fixed point is x* = c/(1 − r) (when r ≠ 1), and x_t − x* = r^t (x₀ − x*). Shifting by the fixed point turns "multiply and add" into just "multiply".

**PREDICT:** what happens if the swarm starts with R₀ = 10? With R₀ = 9?

### BUILD 3: `simulate`
`rec` is a function that maps today's state to tomorrow's. Return the list [x₀, x₁, …, x_steps] (so `steps + 1` entries). The state can be a number or a tuple.

In [ ]:
def simulate(rec, x0, steps):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(simulate, [
    ((lambda x: 2 * x - 10, 15, 3), [15, 20, 30, 50]),
    ((lambda x: x / 2, 8, 3), [8, 4, 2, 1]),
    ((lambda s: (s[0] + s[1], s[0]), (1, 0), 4), [(1, 0), (1, 1), (2, 1), (3, 2), (5, 3)]),
    ((lambda x: x + 1, 0, 0), [0]),
])

### EXPLORE: three behaviours
All three rules below have fixed point 10, and all start at 15. **PREDICT** for each: does x_t settle at 10, run away from it, or bounce around it?
- r = 0.5: x → 0.5x + 5
- r = −0.5: x → −0.5x + 15
- r = 1.1: x → 1.1x − 1

In [ ]:
if prep.ready(simulate, lambda x: x, 0, 1):
    plt.figure(figsize=(6, 3.5))
    for r in [0.5, -0.5, 1.1]:
        c = 10 * (1 - r)
        traj = simulate(lambda x: r * x + c, 15, 15)
        plt.plot(traj, marker=".", label=f"r = {r}")
    plt.axhline(10, color="gray", ls="--", lw=0.8)
    plt.xlabel("step t"); plt.legend(); plt.grid(alpha=.3); plt.title("x_t - 10 = r^t (x_0 - 10)"); plt.show()

The distance to the fixed point gets multiplied by r every step. |r| < 1 shrinks it (settles), |r| > 1 grows it (runs away), and a negative r flips the sign each step (bounces).

### Two quantities at once: a matrix
Many exam recurrences track **two** groups. Example: a robot factory has W worker robots and T trainee robots. Each day every worker trains 2 new trainees, and half of the existing trainees graduate to workers:

W_{t+1} = W_t + ½T_t, T_{t+1} = 2W_t + ½T_t.

Both right-hand sides use **today's** values. The trap is to update W first and then use the *new* W to compute T. Starting from W = 2, T = 4:

In [ ]:
W, T = 2, 4
W2, T2 = 2, 4
rows = [[0, W, T, W2, T2]]
for t in range(1, 4):
    W, T = W + T / 2, 2 * W + T / 2      # simultaneous: both use today's values
    W2 = W2 + T2 / 2                     # trap: W2 is overwritten first...
    T2 = 2 * W2 + T2 / 2                 # ...so T2 uses tomorrow's W2
    rows.append([t, W, T, W2, T2])
prep.table(rows, header=["day", "W (right)", "T (right)", "W (trap)", "T (trap)"])

By day 3 the trap has more than doubled the count. In Python, `W, T = W + T/2, 2*W + T/2` is safe because the whole right side is computed before anything is assigned.

A two-variable linear rule is a **matrix** times a vector. A **matrix** is a grid of numbers. Multiplying a 2 × 2 matrix by a pair (W, T) means: each row of the matrix, times (W, T) entry by entry, added up. Tiny example: [[1, 2], [3, 4]] times (1, 1) is (1 + 2, 3 + 4) = (3, 7).

In [ ]:
M = np.array([[1, 0.5],
              [2, 0.5]])
v = np.array([2, 4])
for t in range(1, 4):
    v = M @ v
    print(f"day {t}: (W, T) = {v}")

Same numbers as the "right" columns. Writing the rule as a matrix makes it impossible to fall into the trap: the new pair is computed from the old pair in one go.

**Now do practice problem 5.4 on paper**, then check:

In [ ]:
prep.check("5.4a", ...)   # U_3
prep.check("5.4b", ...)   # the first day t with U_t > 5000

---
## 10. Linear programming: the best is at a corner

**Idea.** You build two robot kits.
- A **Rover** uses 1 battery and 1 sensor, and earns 3 (hundred rupees).
- An **Arm** uses 1 battery and 3 sensors, and earns 4.

You have 7 batteries and 14 sensors. With x Rovers and y Arms, the **constraints** are x + y ≤ 7 (batteries), x + 3y ≤ 14 (sensors), and x, y ≥ 0. The profit is 3x + 4y.

- The **feasible region** is the set of all (x, y) that satisfy every constraint. Each constraint keeps one side of a straight line, so the region is a polygon.
- A **level line** is the set of points with the same profit, e.g. 3x + 4y = 12. Different profits give parallel lines.

**Why the best is at a corner.** Slide the level line in the direction of higher profit. It keeps touching the region until it's about to leave it, and the last point it touches is a corner (or a whole edge, if the line is parallel to that edge, in which case a corner ties for best).

In [ ]:
xs = np.linspace(0, 8, 200)
plt.figure(figsize=(5.5, 4))
plt.plot(xs, 7 - xs, label="batteries: x + y = 7")
plt.plot(xs, (14 - xs) / 3, label="sensors: x + 3y = 14")
yy = np.minimum(7 - xs, (14 - xs) / 3)
plt.fill_between(xs, 0, np.clip(yy, 0, None), alpha=0.2, label="feasible region")
for level in [12, 18, 24]:
    plt.plot(xs, (level - 3 * xs) / 4, ":", color="gray")
    plt.text(0.1, level / 4 + 0.1, f"profit {level}", fontsize=8, color="gray")
plt.xlim(0, 8); plt.ylim(0, 7.5); plt.xlabel("x (Rovers)"); plt.ylabel("y (Arms)")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

**PREDICT:** which corner of the shaded region does the profit line touch last? Read off its coordinates as well as you can.

A **corner** is where two boundary lines cross. With two constraints plus the two axes there are only a handful of pairs to try.

### BUILD 4: `lp_corners`
`constraints` is a list of (a, b, c), each meaning a·x + b·y ≤ c. The constraints x ≥ 0 and y ≥ 0 are always included. Intersect every pair of boundary lines (including x = 0 and y = 0), skip parallel pairs, keep the points that satisfy **all** constraints, and return the distinct points as a sorted list of (x, y) tuples.

In [ ]:
def lp_corners(constraints):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(lp_corners, [
    (([(1, 1, 4)],), [(0.0, 0.0), (0.0, 4.0), (4.0, 0.0)]),
    (([(1, 1, 7), (1, 3, 14)],), [(0.0, 0.0), (0.0, 14 / 3), (3.5, 3.5), (7.0, 0.0)]),
    (([(1, 0, 2), (0, 1, 2), (1, 1, 4)],), [(0.0, 0.0), (0.0, 2.0), (2.0, 0.0), (2.0, 2.0)]),
])

In [ ]:
if prep.ready(lp_corners, [(1, 1, 1)]):
    kits = [(1, 1, 7), (1, 3, 14)]
    prep.table([[f"({x:.2f}, {y:.2f})", 3 * x + 4 * y] for x, y in lp_corners(kits)], header=["corner", "profit 3x + 4y"])

The best corner is (3.5, 3.5), with profit 24.5. But you can't build half a kit.

**PREDICT:** what's the best plan with **whole** kits? Is it the rounded corner (4, 4)? The rounded-down corner (3, 3)?

### BUILD 5: `best_integer`
Try every whole-number (x, y) with 0 ≤ x, y ≤ limit that satisfies all constraints. Return `(x, y, profit)` for the best. On a tie keep the first found, scanning x upwards and, for each x, y upwards.

In [ ]:
def best_integer(constraints, profit, limit):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(best_integer, [
    (([(1, 1, 7), (1, 3, 14)], lambda x, y: 3 * x + 4 * y, 20), (4, 3, 24)),
    (([(1, 1, 4)], lambda x, y: x + 2 * y, 10), (0, 4, 8)),
    (([(1, 0, 3), (0, 1, 2)], lambda x, y: x + y, 10), (3, 2, 5)),
])

(4, 4) breaks the battery limit, and (3, 3) only earns 21. The best whole-number plan is (4, 3), earning 24. It's a **neighbour** of the corner, but not the one you get by rounding. **Aha: the fractional optimum is at a corner, but the whole-number optimum has to be searched for among the nearby whole-number points**, checking each one's feasibility and profit.

**Now do practice problem 5.5 on paper**, then check:

In [ ]:
prep.check("5.5a", ...)   # the best profit at a corner, with fractional kits allowed
prep.check("5.5b", ...)   # the best profit with whole kits

---
## 11. Base-b place value

**Idea.** In base 10, "345" means 3·100 + 4·10 + 5·1: each place is worth 10 times the place to its right. In **base b**, each place is worth b times the next one. So "1200" in base 3 means 1·27 + 2·9 + 0·3 + 0·1 = 45. Only the digits 0 to b − 1 are allowed.

**Converting to base b.** Divide by b. The remainder is the last digit (the ones place), and the quotient holds all the other digits. Repeat on the quotient, and read the remainders **from the bottom up**. Why it works: n = b·q + r, and multiplying q by b is the same as shifting its digits one place left.

**Smallest example.** 45 in base 3.

In [ ]:
n0, b0 = 45, 3
rows, m = [], n0
while m > 0:
    rows.append([m, f"{m} = {b0} x {m // b0} + {m % b0}", m % b0])
    m //= b0
prep.table(rows, header=["n", "division", "remainder (digit)"])
print("read remainders bottom-up:", "".join(str(r[2]) for r in reversed(rows)))
print("check with Python's int(s, base):", int("1200", 3))

**PREDICT:** in base b, what number does "100" stand for? And "11"? Write each as an expression in b. (These two shapes come back in practice problem 5.6.)

### BUILD 6: `to_base`
Return the base-b digits of a non-negative whole number n as a string, for 2 ≤ b ≤ 10. Zero is "0".

In [ ]:
def to_base(n, b):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(to_base, [
    ((45, 3), "1200"),
    ((5, 2), "101"),
    ((0, 7), "0"),
    ((100, 10), "100"),
    ((8, 8), "10"),
])

**Now do practice problem 5.6 on paper**, then check:

In [ ]:
prep.check("5.6a", ...)   # 2026 in base 7, as a string of digits
prep.check("5.6b", ...)   # the base b

---
## 12. Reading a model off a graph

**Idea.** Exam models often look like this:

$$P(t) = A + B\,t + E \sin\!\Big(\frac{2\pi t}{T}\Big) + D \cdot h(t - t_0)$$

Two pieces need definitions:
- **sin** is a wave that repeats. sin(2πt/T) repeats every T time units (the **period**). Over one period it goes 0 → +1 → 0 → −1 → 0, at t = 0, T/4, T/2, 3T/4 and T.
- **h** is the **step function**: h(s) = 1 if s ≥ 0, else 0. So D·h(t − t₀) is 0 before t₀ and D from t₀ on: a sudden jump of size D.

Here's where the sine sits at the key points of one period, for T = 12:

In [ ]:
rows = [[t, f"2*pi*{t}/12", round(math.sin(2 * math.pi * t / 12), 4) + 0.0] for t in [0, 3, 6, 9, 12, 15]]
prep.table(rows, header=["t", "angle", "sin"])

**The aha: each parameter is controlled by one feature of the graph** (move **M13: read parameters off a graph**).

| Feature of the graph | Gives you |
|---|---|
| distance between two repeats (peak to next peak) | the period T |
| the change between two points one period apart (same sine value, so the sine cancels) | B, from (change) ÷ T |
| a point where sin = +1 and one where sin = −1 | A and E, from two equations |
| the size of the jump | D |

**Smallest example.** A made-up order curve with five labelled points. **PREDICT:** read off T, B, A, E and D yourself, on paper, before running the cell after the plot.

In [ ]:
def model(t, A=6, B=2, E=5, T=12, D=-8, t0=18):
    return A + B * t + E * math.sin(2 * math.pi * t / T) + D * (t >= t0)

ts = np.linspace(0, 24, 600)
labels = [(3, model(3)), (9, model(9)), (15, model(15)), (17.9999, model(17.9999)), (18, model(18))]
plt.figure(figsize=(6, 3.5))
plt.plot(ts[ts < 18], [model(t) for t in ts[ts < 18]], color="C0")
plt.plot(ts[ts >= 18], [model(t) for t in ts[ts >= 18]], color="C0")
for t, v in labels:
    plt.plot([t], [v], "o", color="C3")
    name = "18-" if t < 18 and t > 17 else ("18+" if t == 18 else f"{t:g}")
    plt.annotate(f"({name}, {v:.0f})", (t, v), textcoords="offset points", xytext=(-15, 8), fontsize=8)
plt.xlabel("t (weeks)"); plt.ylabel("P(t)"); plt.grid(alpha=.3); plt.title("Read the parameters off this curve"); plt.show()

**Struggle point:** the point at t = 9 is a *trough* of the sine, yet it's higher than the *peak* at t = 3. How can that be? (Think about what B·t has done in the meantime.)

In [ ]:
print("T: t = 3 and t = 15 are one wave apart, so T =", 15 - 3)
print("B: same sine value at t = 3 and t = 15, so B = (41 - 17) / 12 =", (41 - 17) / 12)
print("At t = 3, sin = +1:  A + 2*3 + E = 17  ->  A + E = 11")
print("At t = 9, sin = -1:  A + 2*9 - E = 19  ->  A - E = 1")
print("Add and subtract:  A = 6, E = 5")
print("D: the jump at t = 18 is 34 - 42 =", 34 - 42)
print("Check a point you didn't use: P(0) = A =", model(0))

The trend climbs 2 per week, so in the 6 weeks from the peak (t = 3) to the trough (t = 9) it rises 12, more than the 10-unit swing of the sine (from +5 to −5). **The labelled points are not necessarily where the curve looks highest or lowest**: the tilt from B·t shifts the visible bumps. Always plug the given t-values into the formula.

### BUILD 7: `fit_from_points`
Once T is known, the model before the step is A·1 + B·t + E·sin(2πt/T): **linear** in the three unknowns A, B, E. Three points give three equations, a **linear system**, which `np.linalg.solve(X, y)` solves. Tiny example: 2a + b = 5 and a − b = 1 is `np.linalg.solve([[2, 1], [1, -1]], [5, 1])`, which gives a = 2, b = 1.

Each row of X holds the numbers multiplying A, B and E for one point. Return `(A, B, E)` as a tuple of three plain floats. (Use `float(...)`: a numpy array won't pass the tests.)

In [ ]:
print(np.linalg.solve([[2, 1], [1, -1]], [5, 1]))

In [ ]:
def fit_from_points(points, T):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(fit_from_points, [
    (([(3, 17), (9, 19), (15, 41)], 12), (6.0, 2.0, 5.0)),
    (([(0, 7), (1, 8), (3, 2)], 4), (7.0, -1.0, 2.0)),
])

**EXPLORE. PREDICT:** what goes wrong if you fit using three points where the sine is 0 (t = 0, 6, 12 when T = 12)? The true values are A = 6, B = 2, E = 5.

In [ ]:
if prep.ready(fit_from_points, [(0, 1), (1, 2), (3, 4)], 4):
    sin0 = [(t, model(t)) for t in (0, 6, 12)]
    print("the E column of X:", [round(math.sin(2 * math.pi * t / 12), 12) + 0.0 for t, _ in sin0])
    try:
        print("fitted (A, B, E):", fit_from_points(sin0, 12))
    except np.linalg.LinAlgError as err:
        print("numpy refused:", err)

The E column is all zeros, so E is multiplied by 0 in every equation. Any E fits those three points equally well, and whatever number the solver hands back for E carries no information. To pin down E, use points where the sine is far from 0, ideally +1 and −1. (In the language of s = Xw: a column of X that is all zeros means its weight can't be learned from that data.)

Now practice problem 5.7. Here is its graph:

In [ ]:
img = plt.imread("../practice/img/5-7-curve.png")
plt.figure(figsize=(8, 4))
plt.imshow(img); plt.axis("off"); plt.show()

Solve it on paper first. Then type your five parameters below: the check compares your model with every labelled point on the graph (without telling you the parameters).

In [ ]:
my_57 = {"A": ..., "B": ..., "E": ..., "T": ..., "D": ...}

def check_57(p):
    if any(v is ... for v in p.values()):
        print("Fill in all five parameters first."); return
    P = lambda t: p["A"] + p["B"] * t + p["E"] * math.sin(2 * math.pi * t / p["T"]) + p["D"] * (t >= 12)
    for t, label, v in [(2, "2", 13), (6, "6", 11), (10, "10", 17), (11.99999, "12-", 16), (12, "12+", 10), (14, "14", 9)]:
        print(f"t = {label:>3}: graph says {v:>4}, your model gives {P(t):7.3f}", "ok" if abs(P(t) - v) < 0.01 else "<-- mismatch")

check_57(my_57)

In [ ]:
prep.check("5.7", ...)   # floor(P(25))

---
## 13. Exam walkthroughs (Jan 18)

> Do these **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf` at the pages given for each task.

### 13a. Recursive V-67 Outbreak (pages 4–5)
Normal infectious N_t, mutant infectious M_t and removed R_t. Read the "each day proceeds in this order" list on page 4 twice. The order is the whole problem.

**Step 1: write the update rule.** On paper, write N_{t+1} and M_{t+1} in terms of N_t and M_t (and R_{t+1} in terms of all three). Be careful about **who recovers**: half of those who were infectious at the **start** of the day, and nobody who was infected that day.

In [ ]:
# your attempt here: write a function v67_step(state) that maps (N, M, R) to the next day's (N, M, R)

<details><summary>Hint 1</summary>

Tomorrow's normal infectious people come from two places: today's normal infectious who did *not* recover, and today's new infections that became normal.

</details>

<details><summary>Hint 2</summary>

X_t = 2N_t + 4M_t. Then N_{t+1} = ½N_t + ½X_t and M_{t+1} = ½M_t + ¼X_t. Expand X_t and collect terms.

</details>

<details><summary>Hint 3</summary>

Two variables updated together: write it as a matrix [[a, b], [c, d]] times (N, M), as in section 9. Then use `simulate` (or the matrix) for three days, and add N₃ + M₃.

</details>

**Step 2: check your rule against facts in the story.** Every newly infected person ends up in N, M or R, and recoveries only move people from N or M into R. So each day, N + M + R must grow by **exactly** X_t = 2N_t + 4M_t. And if nobody is infectious, nothing should change. Paste your step function name into the call below.

In [ ]:
def check_v67_step(step):
    try:
        step((1, 1, 0))
    except Exception as err:
        print("Write your step function first (it should take (N, M, R) and return the next (N, M, R)).", type(err).__name__); return
    ok = True
    for N, M, R in [(8, 0, 0), (0, 8, 0), (4, 6, 10), (10, 2, 3)]:
        N2, M2, R2 = step((N, M, R))
        grew, X = (N2 + M2 + R2) - (N + M + R), 2 * N + 4 * M
        if abs(grew - X) > 1e-9:
            ok = False
            print(f"from {(N, M, R)}: N+M+R grew by {grew}, but X = {X} new people were infected")
    if tuple(step((0, 0, 5))) != (0, 0, 5):
        ok = False
        print("with nobody infectious, the state should not change")
    print("both facts hold" if ok else "fix the rule and run again")

# check_v67_step(v67_step)   # uncomment once you've written v67_step

In [ ]:
prep.check("J18-virus", ...)   # option letter (or the number N3 + M3)

### 13b. The Phulera Godown Showdown (pages 6–7)
Four fields R₁(40, 180), R₂(120, 80), R₃(200, 140), R₄(80, 20). Two officials pick a godown with two different objectives, and you need the distance between their choices.

**Step 1: Sachivji's godown (L1).** His objective is a sum of |x − xᵢ| + |y − yᵢ|. Which section of this notebook does that sum belong to? Find the best x and the best y. With four fields, read the paper's **note** about ties carefully (move **M1**).

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

The sum splits into an x-part and a y-part (section 5). Minimise each on its own: each is a 1-D "total distance" problem (section 2).

</details>

<details><summary>Hint 2</summary>

With four values, every point between the two middle values is an argmin: a flat bottom. The note says to take the **mean of the range** of possible coordinates, i.e. the midpoint of that interval.

</details>

<details><summary>Hint 3</summary>

If you check with `argmin_brute` over a grid, it returns the *first* point of the flat bottom, not the midpoint. Why? Which one does the paper want?

</details>

**Step 2: Pradhanji's godown (the divergence).** His objective is

$$\sum_i \Big[x_i \ln\frac{x_i}{u} + y_i \ln\frac{y_i}{v} + (u - x_i) + (v - y_i)\Big]$$

Two new things here, so take them one at a time.

**(a) It's separable again.** The u-terms and the v-terms don't mix, so find the best u using only the xᵢ, and the best v using only the yᵢ.

**(b) ln.** ln is the **natural logarithm**: ln(z) is the power you raise e ≈ 2.718 to in order to get z, just as log₂ uses 2 (notebook 04). So ln(1) = 0, ln(e) = 1, and ln(a/b) = ln a − ln b.

**(c) Find the bottom of the valley through the slope.** From section 6: the slope at u is the change in f per unit change in u, over a tiny step, and **at the bottom of a valley the slope is 0**. One fact about ln is all you need: for a tiny step ε, ln(u + ε) − ln u ≈ ε/u. So the slope of ln u is **1/u**. Check that numerically:

In [ ]:
eps = 1e-6
rows = [[u, round((math.log(u + eps) - math.log(u)) / eps, 4), 1 / u] for u in [0.5, 1, 2, 5, 10]]
prep.table(rows, header=["u", "slope of ln u (tiny step)", "1/u"])

Now one term of Pradhanji's sum, as a function of u, with the field's coordinate x fixed:

f(u) = x ln(x/u) + u − x = x ln x − x ln u + u − x.

- x ln x and −x don't depend on u, so they don't change the slope.
- −x ln u has slope −x · (1/u).
- u has slope 1.

So the slope of f is **1 − x/u**. Check it numerically for x = 4:

In [ ]:
x_f = 4
f_one = lambda u: x_f * math.log(x_f / u) + u - x_f
rows = [[u, round((f_one(u + eps) - f_one(u)) / eps, 4) + 0.0, 1 - x_f / u] for u in [1, 2, 4, 8]]
prep.table(rows, header=["u", "slope of f (tiny step)", "1 - x/u"])

The slope is negative left of u = 4 (going right goes downhill), zero at u = 4, and positive after (uphill). One term alone is happiest at u = x.

**PREDICT / derive:** with several fields, the slope of the whole u-part is the sum of the slopes of its terms. Add up (1 − xᵢ/u) over n fields, set it to 0, and solve for u. Which familiar quantity do you get? Test your formula on the toy values [2, 4, 9] in the cell below (a brute-force search, not the godown data).

In [ ]:
toy = [2, 4, 9]
if prep.ready(argmin_brute, abs, [1]):
    kl_u = lambda u: sum(xi * math.log(xi / u) + u - xi for xi in toy)
    print("brute-force best u for the toy values:", argmin_brute(kl_u, [round(v, 2) for v in np.arange(0.5, 12, 0.01)]))

<details><summary>Hint</summary>

Σ(1 − xᵢ/u) = n − (Σxᵢ)/u. Setting that to 0 gives u = (Σxᵢ)/n. Is the slope negative to the left of that point and positive to the right (a valley, not a hill)?

</details>

**Step 3: the distance.** Now compute Pradhanji's (u, v) for the four fields, then D = √((x_S − x_P)² + (y_S − y_P)²), and report ⌊D⌋.

In [ ]:
# your attempt here

In [ ]:
prep.check("J18-godown", ...)

**Connect it back:** L1 picked the **median** in each coordinate, and this divergence picked the **mean**. That's the same split as MAE vs RMSE in notebook 01 and 5.1(a) vs 5.1(b) here.

### 13c. The Crude Curve Forecast (pages 7–8)
P(t) = A + Bt + C·(1 − e^{−(t−τ_C)})·h(t − τ_C) + D·h(t − τ_D) + E·sin(2πt/τ_E), and a graph for 0 ≤ t ≤ 60. You want ⌊P(75)⌋.

**Step 0: read the formula, not the prose** (move **M1**). The sentence listing the unknowns on page 8 has typos (it mentions T, τ_A, τ_B, which don't appear in the formula). The formula is what counts: the unknowns are A, B, C, D, E, τ_C, τ_D and τ_E.

Match each unknown to one graph feature, as in section 12. Take them in this order.

**Step 1: the period τ_E.** Which labelled points before t = 30 are one wave apart?

**Step 2: the trend B.** Use two points one period apart, so the sine cancels.

**Step 3: A and E.** Use a labelled peak (sin = +1) and a labelled trough (sin = −1) before t = 30, where neither step has kicked in. Check A against P(0) on the graph.

**Step 4: τ_D and D.** Where does the curve jump, and by how much?

**Step 5: τ_C and C.** The second dashed line marks τ_C. A new function appears here: e^{−s} (e ≈ 2.718 to the power −s) **shrinks fast**: e^{−1} ≈ 0.37, e^{−3} ≈ 0.05, e^{−9} ≈ 0.0001. So a few months after τ_C, the C-term is almost exactly C.

**Step 6: P(75).** Which terms are "fully switched on" by t = 75, and where is the sine?

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

The peaks at t = 3, 15 and 27 are labelled. The gap between neighbours is the period. The same gap gives you B from the rise between them.

</details>

<details><summary>Hint 2</summary>

At t = τ_C exactly, 1 − e⁰ = 0, so the C-term is 0 there. The labelled point at t = 48 therefore can't tell you C. It's a **check** on everything else.

</details>

<details><summary>Hint 3</summary>

To find C, read one value off the graph well after t = 48, where the e-term has died out. The trough near t = 57 is a good choice: what would the model give there **without** C, and what does the graph show?

</details>

Type your parameters to check them against the graph's labelled points. The cell also prints your model at t = 51 and t = 57: compare those with what you read off the figure.

In [ ]:
my_crude = {"A": ..., "B": ..., "C": ..., "D": ..., "E": ..., "tau_C": ..., "tau_D": ..., "tau_E": ...}

def check_crude(p):
    if any(v is ... for v in p.values()):
        print("Fill in all eight parameters first."); return
    def P(t):
        c_term = p["C"] * (1 - math.exp(-(t - p["tau_C"]))) if t >= p["tau_C"] else 0.0
        return (p["A"] + p["B"] * t + c_term + p["D"] * (t >= p["tau_D"])
                + p["E"] * math.sin(2 * math.pi * t / p["tau_E"]))
    for t, label, v in [(3, "3", 34), (9, "9", 14), (15, "15", 42), (21, "21", 22), (27, "27", 50),
                        (29.99999, "30-", 40), (30, "30+", 60), (48, "48", 72)]:
        print(f"t = {label:>3}: graph says {v:>3}, your model gives {P(t):7.2f}", "ok" if abs(P(t) - v) < 0.05 else "<-- mismatch")
    print(f"not labelled, read these off the figure yourself:  P(51) = {P(51):.1f},  P(57) = {P(57):.1f}")
    tt = np.linspace(0, 60, 601)
    plt.figure(figsize=(6, 3)); plt.plot(tt, [P(t) for t in tt]); plt.grid(alpha=.3)
    plt.title("your model: compare its shape with Figure 3"); plt.xlabel("t (months)"); plt.show()

check_crude(my_crude)

In [ ]:
prep.check("J18-crude", ...)

### 13d. The Catalyst Constraint (pages 8–9)
d_A = a·c_A − 2c_A² and d_B = b·c_B − 3c_B², with a = 16, b = 24 and c_A + c_B ≤ 5 moles of catalyst. Maximise d_A + d_B.

**Step 1: does the budget bind?** Find each pathway's own best c (its vertex, section 3). Do the two together need more than 5 moles?

**Step 2: if it binds, equalise the marginal gains.** The slope of a·c − 2c² is a − 4c (section 6: the slope of pc² + qc is 2pc + q). Write the slope of d_B too, set the two equal, and use c_A + c_B = 5.

**Step 3: compute the total yield and round down.**

In [ ]:
# your attempt here

<details><summary>Hint 1</summary>

The vertex of pc² + qc is at c = −q/(2p). For pathway A, p = −2 and q = 16. Do the same for pathway B, add the two, and compare with 5.

</details>

<details><summary>Hint 2</summary>

It binds, so the whole 5 moles get used. Set 16 − 4c_A = 24 − 6c_B and substitute c_B = 5 − c_A. Watch for the trap: the *marginal gains* are equal at the optimum, not the yields.

</details>

<details><summary>Hint 3</summary>

Check your answer with `best_split_budget` using a step of 0.01. Do the two agree?

</details>

In [ ]:
prep.check("J18-catalyst", ...)

---
## 14. Connect

Answer in your own words in the cell below. Then copy the best sentence into `log/ERROR_LOG.md` under "Moves I own".

1. In notebook 01 (section 7), MAE was minimised by the median and RMSE by the mean. Here, 5.1 and the Godown gave the same split. Explain in two sentences why "absolute error → median" and "squared error → mean" are the same facts you proved in sections 2 and 3. Then: k-means (notebook 03) moves each centroid to the **mean** of its points. Which loss is it minimising?
2. PageRank (notebook 04) and the recurrences here both repeat an update until something settles. For x → r·x + c, the distance to the fixed point is multiplied by r each step. What plays the role of r in PageRank, and why does that guarantee it settles?
3. Fitting A, B and E in section 12 meant solving X·w = y for the weights w, and WonkaRank's Score A (notebook 04) was a weights-times-features sum. In both, what are the rows of X and what is w? What went wrong when the three points all had sin = 0?

*Your answers here.*